In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.datasets as datasets
from collections import Counter
from pathlib import Path

In [ ]:
# ---------- 固定随机种子 ----------
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
# ---------- 路径与参数 ----------
BASE_DIR = Path().cwd().parent
DATA_ROOT = BASE_DIR/'data'
OUTPUT_DIR = BASE_DIR/'outputs'
CSV_FILE = 'cifar10_inventory.csv'
CLASS_COUNT_PLOT = 'class_count.png'
BRIGHTNESS_HIST_PLOT = 'brightness_hist.png'
SAMPLE_GRID_PLOT = 'sample_grid.png'

CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']
N_CLASSES = len(CLASSES)
SUB_SAMPLES_PER_CLASS = 100   # 子集每类抽取100张
GRID_PER_CLASS = 16           # 拼图每类16张 (4x4)

In [ ]:
# ---------- 1. 加载CIFAR-10训练集（原始PIL图像）----------
print("正在加载CIFAR-10训练集...")
# 不应用变换，直接得到PIL Image
trainset = datasets.CIFAR10(root=DATA_ROOT, train=True, download=False)
images = []          # 存放所有PIL图像
labels = []          # 存放所有标签（整数）
for img, label in trainset:
    images.append(img)
    labels.append(label)
print(f"加载完成，共 {len(images)} 张图像。")

In [ ]:
# ---------- 2. 统计整体数据集信息 ----------
print("\n--- 整体数据集统计 ---")
# 类别数量
# class_counts = Counter(labels)
class_counts = [labels.count(i) for i in range(len(CLASSES))]
print("各类别样本数：")
for cls_id, cls_name in enumerate(CLASSES):
    print(f"  {cls_name}: {class_counts[cls_id]}")

# 尺寸、通道数、dtype
sample_img = images[0]
width, height = sample_img.size  # PIL Image 尺寸 (width, height)
channels = len(sample_img.getbands())  # 通道数
dtype = np.array(sample_img).dtype  # uint8
print(f"图像尺寸: {width} x {height}, 通道数: {channels}, dtype: {dtype}")

# 整体均值和标准差（按通道，像素值范围0-255）
# 将所有图像堆叠成 (N, H, W, C) 的numpy数组，uint8，转换为float32计算
print("正在计算整体均值和标准差...")
all_imgs_np = np.stack([np.array(img) for img in images]).astype(np.float32)  # (50000,32,32,3)
mean_overall = all_imgs_np.mean(axis=(0, 1, 2))   # 三个通道均值
std_overall = all_imgs_np.std(axis=(0, 1, 2))     # 三个通道标准差
print(f"整体均值 (R, G, B): {mean_overall}")
print(f"整体标准差 (R, G, B): {std_overall}")

In [ ]:

# ---------- 3. 构建子集（每类随机抽取100张）----------
print("\n构建子集（每类100张，随机种子42）...")
subset_indices = []   # 存放选取的全局索引
for cls_id in range(N_CLASSES):
    # 获取该类别所有样本的索引
    cls_indices = [i for i, lab in enumerate(labels) if lab == cls_id]
    # 随机抽取100个，使用固定种子保证可复现
    selected = random.sample(cls_indices, SUB_SAMPLES_PER_CLASS)
    subset_indices.extend(selected)

# 根据索引构建子集图像和标签
subset_images = [images[i] for i in subset_indices]
subset_labels = [labels[i] for i in subset_indices]

# 验证每类是否恰好100张
subset_class_counts = Counter(subset_labels)
print("子集各类别样本数（应为100）：")
for cls_id, cls_name in enumerate(CLASSES):
    print(f"  {cls_name}: {subset_class_counts[cls_id]}")

In [ ]:

# ---------- 4. 生成CSV（子集记录）----------
print("\n生成CSV...")
csv_data = []
for idx, (img, lab) in enumerate(zip(subset_images, subset_labels)):
    arr = np.array(img)  # (32,32,3) uint8
    mean_r, mean_g, mean_b = arr[:,:,0].mean(), arr[:,:,1].mean(), arr[:,:,2].mean()
    std_r, std_g, std_b = arr[:,:,0].std(), arr[:,:,1].std(), arr[:,:,2].std()
    csv_data.append({
        'index': idx,
        'class_id': lab,
        'class_name': CLASSES[lab],
        'width': width,
        'height': height,
        'channels': channels,
        'dtype': str(dtype),   # 'uint8'
        'mean_r': mean_r,
        'mean_g': mean_g,
        'mean_b': mean_b,
        'std_r': std_r,
        'std_g': std_g,
        'std_b': std_b
    })

df = pd.DataFrame(csv_data)
df.to_csv(OUTPUT_DIR/CSV_FILE, index=False)
print(f"CSV已保存至: {CSV_FILE}")

In [ ]:

# ---------- 5. 绘制整体类别分布柱状图 ----------
print("\n绘制类别分布图...")
plt.figure(figsize=(10, 6))
plt.bar(CLASSES, [class_counts[i] for i in range(N_CLASSES)], color='skyblue')
plt.xlabel('CLASS', fontsize=12)
plt.ylabel('AMOUNT', fontsize=12)
plt.title('CIFAR-10 trainset class distribution (50,000)', fontsize=14)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/CLASS_COUNT_PLOT, dpi=300, bbox_inches='tight')
plt.close()
print(f"类别分布图已保存: {OUTPUT_DIR/CLASS_COUNT_PLOT}")

In [ ]:
# ---------- 6. 绘制整体亮度直方图 (bins=50) ----------
# 使用增量直方图，避免存储所有像素值
print("正在计算亮度直方图（增量方式）...")
hist = None
for i, img in enumerate(images):
    arr = np.array(img).astype(np.float32)  # (32,32,3)
    # 计算亮度（加权灰度）
    gray = 0.299 * arr[:,:,0] + 0.587 * arr[:,:,1] + 0.114 * arr[:,:,2]
    # 第一次创建直方图，后续累计
    if hist is None:
        hist, bin_edges = np.histogram(gray, bins=50, range=(0, 255))
    else:
        hist += np.histogram(gray, bins=50, range=(0, 255))[0]
    # 每1000张打印一次进度
    if (i+1) % 1000 == 0:
        print(f"已处理 {i+1} 张图像")

# 绘制直方图
plt.figure(figsize=(10, 6))
plt.bar(bin_edges[:-1], hist, width=np.diff(bin_edges), color='gray', alpha=0.7, edgecolor='black')
plt.xlabel('brightness', fontsize=12)
plt.ylabel('px', fontsize=12)
plt.title('CIFAR-10 trainset brightness hist (bins=50)', fontsize=14)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/BRIGHTNESS_HIST_PLOT, dpi=300, bbox_inches='tight')
plt.close()
print(f"亮度直方图已保存: {OUTPUT_DIR/BRIGHTNESS_HIST_PLOT}")

In [ ]:

# ---------- 7. 生成每类样本拼图 (sample_grid.png) ----------
print("生成样本拼图...")
# 从子集中每类取前16张（顺序固定，也可随机，但用前16保证可复现）
grid_images = []
for cls_id in range(N_CLASSES):
    # 获取子集中该类别的所有图像
    cls_imgs = [img for img, lab in zip(subset_images, subset_labels) if lab == cls_id]
    # 取前16张
    selected = cls_imgs[:GRID_PER_CLASS]
    grid_images.append(selected)

# 定义拼接函数：将图像列表拼成 rows x cols 网格，返回PIL Image
def make_grid(images, rows, cols, padding=2, bg_color=(255,255,255)):
    if not images:
        return None
    # 所有图像尺寸应相同
    w, h = images[0].size
    # 创建背景画布
    grid_w = cols * w + (cols - 1) * padding
    grid_h = rows * h + (rows - 1) * padding
    grid = Image.new('RGB', (grid_w, grid_h), bg_color)
    for i, img in enumerate(images):
        if i >= rows * cols:
            break
        x = (i % cols) * (w + padding)
        y = (i // cols) * (h + padding)
        grid.paste(img, (x, y))
    return grid

# 每个类生成一个4x4小网格，然后垂直拼接10个小网格（每行一个类别）
small_grids = []
for cls_id, imgs in enumerate(grid_images):
    grid = make_grid(imgs, rows=4, cols=4, padding=2)
    # 在顶部添加类别标签（使用PIL绘图）
    from PIL import ImageDraw, ImageFont
    # 创建一个稍宽的画布以容纳标签
    label_height = 20
    label_img = Image.new('RGB', (grid.width, grid.height + label_height), (255,255,255))
    draw = ImageDraw.Draw(label_img)
    # 尝试使用默认字体，若系统无则使用内置
    try:
        font = ImageFont.truetype("arial.ttf", 14)
    except:
        font = ImageFont.load_default()
    draw.text((5, 5), f"{CLASSES[cls_id]}", fill=(0,0,0), font=font)
    label_img.paste(grid, (0, label_height))
    small_grids.append(label_img)

# 垂直拼接所有小网格
total_height = sum(img.height for img in small_grids)
max_width = max(img.width for img in small_grids)
final_grid = Image.new('RGB', (max_width, total_height), (255,255,255))
y_offset = 0
for img in small_grids:
    final_grid.paste(img, (0, y_offset))
    y_offset += img.height

final_grid.save(OUTPUT_DIR/SAMPLE_GRID_PLOT)
print(f"样本拼图已保存: {OUTPUT_DIR/SAMPLE_GRID_PLOT}")

print("\n所有任务完成！")
print(f"生成文件: {CSV_FILE}, {CLASS_COUNT_PLOT}, {BRIGHTNESS_HIST_PLOT}, {SAMPLE_GRID_PLOT}")